# Lab 2 — Ứng dụng Pre-trained Models trên CIFAR-10

> **Môn học:** Deep Learning — **Chủ đề:** Transfer Learning & Fine-tuning bằng PyTorch  
> **Tập dữ liệu:** CIFAR-10 (60.000 ảnh màu 32×32, 10 lớp)

Notebook này điều phối toàn bộ quy trình phân loại CIFAR-10 bằng các mô hình pretrained VGG16 và ResNet-18. Trên Colab, nguồn code được lấy từ GitHub; dữ liệu và kết quả cần lưu lâu dài được đặt trên Google Drive.

## Bản tóm tắt nội dung

| Mục | Nội dung | Phạm vi / trạng thái |
|---|---|---|
| **0. Setup & Run Configuration** | Nguồn code, đường dẫn Drive/runtime, chọn `RUN_ID` | Phase 8: cấu hình và kiểm tra, không huấn luyện |
| **1. CIFAR-10 Data** | Kiểm tra dữ liệu và tạo train/validation/test loaders | Chỉ chạy sau khi cấu hình Drive/xác thực dataset |
| **2. Pretrained Architectures** | Khởi tạo và khám phá kiến trúc | Đã điều khiển bởi `RUN_SPECS[RUN_ID]` |
| **3. CIFAR-10 Model Adaptation** | Classifier, đóng gói và BatchNorm | Đã điều khiển bởi `RUN_SPECS[RUN_ID]` |
| **4. Training & Validation** | Huấn luyện, theo dõi và lưu checkpoint | Chỉ train/reuse cho `RUN_ID` được chọn; không tự chạy thêm run khác |
| **5. Model Comparison** | So sánh lịch sử các run | Thực hiện sau khi có kết quả |
| **6. ResNet-18 Fine-tuning** | Fine-tune từ checkpoint baseline | Thực hiện sau khi baseline hoàn tất |
| **7. Final Test Evaluation** | Đánh giá cuối trên test set | Chỉ sau khi chốt cấu hình; không dùng để chọn hyperparameter |

## Quy ước dữ liệu

- CIFAR-10 được chia thành **45.000 train**, **5.000 validation** và **10.000 test**.
- Train dùng để cập nhật weights; validation dùng để chọn checkpoint/cấu hình.
- Test chỉ dùng để đánh giá cuối sau khi mô hình/run đã được chốt.

> **Giới hạn Phase 8:** Phạm vi hiện tại là chuẩn hóa setup và cấu hình. Các cell experiment phía dưới cần giữ nguyên cho VGG16, ResNet-18 và fine-tune; chúng chưa được điều khiển bởi `RUN_ID` và không chạy tuần tự qua các cell experiment trong Phase 8.



---
## 0. Setup & Run Configuration

Cell setup phân biệt code checkout, nguồn CIFAR-10 trên Drive, thư mục dữ liệu runtime, artifacts và TensorBoard logs. Cell cấu hình tiếp theo chỉ xác thực và hiển thị một `RUN_ID` cùng các tham số tương ứng.

**Trong Phase 8:** không chạy huấn luyện, resume, inference reuse hay đánh giá test. Trên Colab, chỉ thực thi setup sau khi đã đưa phiên bản notebook/code lên GitHub và chuẩn bị cấu trúc Drive theo hướng dẫn.



In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/toan58936/lab2.git"
GITHUB_BRANCH = "main"
GITHUB_COMMIT = None  # Set a commit hash to pin the exact source revision.

try:
    import google.colab
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    CODE_ROOT = Path("/content/lab2")
    DRIVE_ROOT = Path("/content/drive/MyDrive/lab2")
    DRIVE_DATA_ROOT = DRIVE_ROOT / "data"
    DRIVE_ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    DRIVE_RUNS_ROOT = DRIVE_ROOT / "runs"
    DATA_DIR = Path("/content/cifar10_data")

    if not CODE_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REPO_URL, str(CODE_ROOT)],
            check=True,
        )
    elif not (CODE_ROOT / ".git").exists():
        raise FileNotFoundError(
            f"{CODE_ROOT} exists but is not a Git checkout. Rename it before cloning."
        )

    if GITHUB_COMMIT is not None:
        subprocess.run(["git", "-C", str(CODE_ROOT), "checkout", GITHUB_COMMIT], check=True)

    CODE_REVISION = subprocess.run(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    ARTIFACT_ROOT = DRIVE_ARTIFACT_ROOT
    RUNS_ROOT = DRIVE_RUNS_ROOT
else:
    CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_ROOT = None
    DRIVE_DATA_ROOT = None
    DRIVE_ARTIFACT_ROOT = None
    DRIVE_RUNS_ROOT = None
    DATA_DIR = CODE_ROOT / "data"
    ARTIFACT_ROOT = CODE_ROOT / "artifacts"
    RUNS_ROOT = CODE_ROOT / "runs"
    CODE_REVISION = "local working tree"

CODE_ROOT = CODE_ROOT.resolve()
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print(f"CODE_ROOT      : {CODE_ROOT}")
print(f"CODE_REVISION  : {CODE_REVISION}")
print(f"DRIVE_ROOT     : {DRIVE_ROOT}")
print(f"DRIVE_DATA_ROOT: {DRIVE_DATA_ROOT}")
print(f"DATA_DIR       : {DATA_DIR}")
print(f"ARTIFACT_ROOT  : {ARTIFACT_ROOT}")
print(f"RUNS_ROOT      : {RUNS_ROOT}")
print(f"Colab          : {IS_COLAB}")


In [ ]:
import torch
from torch import nn
import torchvision.models as models
import matplotlib.pyplot as plt
import pandas as pd
from timeit import default_timer as timer

try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    raise ImportError('Cài tensorboard: pip install tensorboard')

from going_modular import data_setup, engine, evaluate, model_builder, utils
from torchinfo import summary as torchinfo_summary
from going_modular.model_builder import set_train_mode, count_params

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
DEVICE = utils.get_device()
print(f'DEVICE    : {DEVICE}')


In [ ]:
# ============================================================
# PHASE 8: RESOLVE CONFIGURATION ONLY; TRAINING IS OUT OF SCOPE
# ============================================================
SEED = 42
VAL_SIZE = 5_000
NUM_WORKERS = 0
TRANSFORM_WEIGHTS = "IMAGENET1K_V1"
RUN_MODE = "train"  # "train" or "reuse" for the later execution phase.
RESUME = False       # Only applies to RUN_MODE == "train".

ALLOWED_RUN_IDS = (
    "v0_vgg16",
    "v1_resnet18",
    "v1_resnet18_finetune",
    "v2_resnet18_lr1e-2",
    "v2_resnet18_lr1e-4",
    "v2_resnet18_bs64",
)
RUN_ID = "v0_vgg16"

if RUN_ID not in ALLOWED_RUN_IDS:
    raise ValueError(f"Unknown RUN_ID {RUN_ID!r}; choose one of {ALLOWED_RUN_IDS}.")
if RUN_MODE not in {"train", "reuse"}:
    raise ValueError("RUN_MODE must be 'train' or 'reuse'.")
if RESUME and RUN_MODE != "train":
    raise ValueError("RESUME=True is only valid when RUN_MODE='train'.")

RUN_SPECS = {
    "v0_vgg16": {
        "arch": "vgg16", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-3,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v1_resnet18": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-3,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v1_resnet18_finetune": {
        "arch": "resnet18", "weights": None, "freeze_backbone": True,
        "unfreeze_last_n": 1, "epochs": 3, "batch_size": 32, "lr": 1e-4,
        "optimizer": "adam", "param_groups": {"head": 1e-3, "backbone": 1e-4},
        "dependency_run_id": "v1_resnet18",
        "source_checkpoint": "v1_resnet18/best_model.pth",
    },
    "v2_resnet18_lr1e-2": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-2,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v2_resnet18_lr1e-4": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-4,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v2_resnet18_bs64": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 64, "lr": 1e-3,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
}

spec = RUN_SPECS[RUN_ID]

# ---------------------------------------------------------------------------
# Artifact guards (A.4): kiểm tra trạng thái artifact trước khi chạy
# ---------------------------------------------------------------------------
RUN_ARTIFACT_DIR = ARTIFACT_ROOT / RUN_ID
RUN_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN_ARTIFACT_DIR / "config.json"
HISTORY_PATH = RUN_ARTIFACT_DIR / "history.json"
BEST_MODEL_PATH = RUN_ARTIFACT_DIR / "best_model.pth"
LAST_CHECKPOINT_PATH = RUN_ARTIFACT_DIR / "last_checkpoint.pth"
TEST_METRICS_PATH = RUN_ARTIFACT_DIR / "test_metrics.json"

def _artifact_files_present():
    return all(p.is_file() for p in [CONFIG_PATH, HISTORY_PATH, BEST_MODEL_PATH])

if RUN_MODE == "train" and not RESUME:
    # Train mới: không tự ghi đè run đã có artifacts đầy đủ
    if _artifact_files_present():
        raise FileExistsError(
            f"Run '{RUN_ID}' da co artifacts day du tai {RUN_ARTIFACT_DIR}. "
            "Khong tu ghi de. Dung RESUME=True de tiep tuc hoac RUN_MODE='reuse' de nap lai."
        )
elif RUN_MODE == "train" and RESUME:
    # Resume: yeu cau last_checkpoint.pth ton tai va doc duoc
    if not LAST_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(
            f"RESUME=True ma thieu last_checkpoint.pth: {LAST_CHECKPOINT_PATH}. "
            "Hay train run nay truoc (RESUME=False) hoac dung RUN_MODE='reuse'."
        )
elif RUN_MODE == "reuse":
    # Reuse: yeu du config/history/best_model
    if not _artifact_files_present():
        missing = [str(p) for p in [CONFIG_PATH, HISTORY_PATH, BEST_MODEL_PATH] if not p.is_file()]
        raise FileNotFoundError(
            f"RUN_MODE='reuse' yeu cau artifacts day du cho '{RUN_ID}' nhung thieu: {missing}. "
            "Hay train run nay truoc."
        )

# Validate dependency (fine-tune yeu cau baseline da co artifact)
if spec["dependency_run_id"] is not None:
    dep_artifact = ARTIFACT_ROOT / spec["dependency_run_id"] / "best_model.pth"
    if RUN_MODE == "train" and not dep_artifact.is_file():
        raise FileNotFoundError(
            f"{RUN_ID} yeu cau checkpoint baseline da hoan thanh: {dep_artifact}"
        )


# ---------------------------------------------------------------------------
# Outputs & logging theo giai dau (dated outputs / session logs)
# ---------------------------------------------------------------------------
OUTPUT_ROOT = ARTIFACT_ROOT.parent / "outputs"
LOG_ROOT = ARTIFACT_ROOT.parent / "logs"
SESSION_ID = utils.make_session_id()
EVAL_ID = utils.make_eval_id()
EVAL_OUTPUT_DIR = utils.make_eval_output_dir(OUTPUT_ROOT, RUN_ID, eval_id=EVAL_ID)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Tham so chay (resolve tu spec)
BATCH_SIZE = spec["batch_size"]
EPOCHS = spec["epochs"]
LR = spec["lr"]

print(f"RUN_ID          : {RUN_ID}")
print(f"RUN_MODE        : {RUN_MODE}")
print(f"RESUME          : {RESUME}")
print(f"SESSION_ID      : {SESSION_ID}")
print(f"EVAL_ID         : {EVAL_ID}")
print(f"Output root     : {OUTPUT_ROOT}")
print(f"Log root        : {LOG_ROOT}")
print(f"Resolved spec   : {spec}")
print(f"Data source     : {DRIVE_DATA_ROOT if IS_COLAB else DATA_DIR}")
print(f"DataLoader path : {DATA_DIR}")
print(f"Run artifacts   : {RUN_ARTIFACT_DIR}")
print(f"Guard passed    : RUN_MODE={RUN_MODE} | RESUME={RESUME}")
print(f"Seed={SEED} | Val_size={VAL_SIZE} | Num_workers={NUM_WORKERS}")



---
## 1. CIFAR-10 Data

Phần này xác thực dữ liệu CIFAR-10 để chuẩn bị trên Drive, có thể stage sang thư mục runtime của Colab, rồi tạo ba DataLoader. Dataset phải có các batch CIFAR-10 chuẩn và `batches.meta`; DataLoader được gọi với `download=False` để không tự tải dữ liệu ngoài cấu hình.

Train và validation được tách theo seed từ tập train gốc; test được giữ riêng cho đánh giá cuối. Việc đọc batch thực thi sau khi người dùng cấu hình Drive, không thuộc bước kiểm tra code của các bậc.



In [ ]:
import shutil
import tarfile
import tempfile
from pathlib import PurePosixPath

_CIFAR_REQUIRED_FILES = {
    "data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4",
    "data_batch_5", "test_batch", "batches.meta",
}


def _valid_cifar10_directory(path: Path) -> bool:
    return path.is_dir() and _CIFAR_REQUIRED_FILES.issubset(
        {item.name for item in path.iterdir() if item.is_file()}
    )


def _stage_cifar10_from_drive() -> None:
    destination = DATA_DIR / "cifar-10-batches-py"
    if _valid_cifar10_directory(destination):
        return
    if destination.exists():
        raise FileExistsError(
            f"Incomplete CIFAR-10 directory at {destination}; it will not be overwritten."
        )

    source_directory = DRIVE_DATA_ROOT / "cifar-10-batches-py"
    if not _valid_cifar10_directory(source_directory) and _valid_cifar10_directory(DRIVE_DATA_ROOT):
        source_directory = DRIVE_DATA_ROOT

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if _valid_cifar10_directory(source_directory):
        shutil.copytree(source_directory, destination)
    else:
        archive_path = DRIVE_DATA_ROOT / "cifar-10-python.tar.gz"
        if not archive_path.is_file():
            raise FileNotFoundError(
                "CIFAR-10 was not found on Drive. Upload either "
                f"{source_directory} or {archive_path}."
            )

        with tempfile.TemporaryDirectory(prefix="cifar10_stage_") as temporary_directory:
            with tarfile.open(archive_path, "r:gz") as archive:
                members = archive.getmembers()
                for member in members:
                    member_path = PurePosixPath(member.name)
                    if (
                        member_path.is_absolute()
                        or ".." in member_path.parts
                        or not member_path.parts
                        or member_path.parts[0] != "cifar-10-batches-py"
                        or not (member.isfile() or member.isdir())
                    ):
                        raise ValueError(
                            f"Unexpected or unsafe member in CIFAR-10 archive: {member.name}"
                        )
                archive.extractall(temporary_directory, members=members)

            extracted_directory = Path(temporary_directory) / "cifar-10-batches-py"
            if not _valid_cifar10_directory(extracted_directory):
                raise ValueError(
                    f"The archive does not contain a valid CIFAR-10 dataset: {archive_path}"
                )
            shutil.copytree(extracted_directory, destination)

    if not _valid_cifar10_directory(destination):
        raise ValueError(f"Staged CIFAR-10 dataset is incomplete: {destination}")


if IS_COLAB:
    _stage_cifar10_from_drive()
elif not _valid_cifar10_directory(DATA_DIR / "cifar-10-batches-py"):
    raise FileNotFoundError(
        f"CIFAR-10 files were not found under {DATA_DIR}; prepare the dataset before loading."
    )

train_dl, val_dl, test_dl, CLASS_NAMES = data_setup.create_train_val_test_dataloaders(
    batch_size=BATCH_SIZE,
    data_dir=str(DATA_DIR),
    transform_model=spec["arch"],
    transform_weights=TRANSFORM_WEIGHTS,
    val_size=VAL_SIZE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    download=False,
)

NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train batches : {len(train_dl):>5}  (~{len(train_dl) * BATCH_SIZE:>6} samples)")
print(f"Val batches   : {len(val_dl):>5}  (~{len(val_dl) * BATCH_SIZE:>6} samples)")
print(f"Test batches  : {len(test_dl):>5}  (~{len(test_dl) * BATCH_SIZE:>6} samples)")


In [ ]:
# -- Kiểm tra shape & dtype của một batch --
images, labels = next(iter(train_dl))
print(f'Batch images : {images.shape}   # [B, C, H, W]')
print(f'Batch labels : {labels.shape}')
print(f'Dtype        : {images.dtype}')
print(f'Pixel range (ImageNet-normalized): [{images.min():.3f}, {images.max():.3f}]')

In [ ]:
# -- De-normalize để hiển thị ảnh --
# Tensor đã normalize theo ImageNet, cần đảo ngược trước khi plt.imshow()
from torchvision.models import VGG16_Weights
WEIGHT_TRANSFORM = VGG16_Weights.IMAGENET1K_V1.transforms()
IMAGENET_MEAN = torch.tensor(WEIGHT_TRANSFORM.mean).view(3, 1, 1)
IMAGENET_STD  = torch.tensor(WEIGHT_TRANSFORM.std).view(3, 1, 1)

def denormalize(t):
    """[C,H,W] normalized → [C,H,W] in [0,1]"""
    return (t * IMAGENET_STD + IMAGENET_MEAN).clamp(0.0, 1.0)

torch.manual_seed(SEED)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img, lbl in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[lbl.item()], fontsize=9)
    ax.axis('off')
fig.suptitle('16 ảnh train (de-normalized, center crop 224×224)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Pretrained Architectures

Phần này khởi tạo và khám phá kiến trúc VGG16, ResNet-18 kèm ImageNet weights, sau đó xem cấu trúc và số lượng tham số. Ở chế độ train, việc tải pretrained weights cần mạng và thuộc giai đoạn thực nghiệm.

Trong Phase 8, không chạy các cell khởi tạo model nếu chưa tải weights; chỉ hoàn thiện cấu hình notebook.



In [ ]:
# ============================================================
# Xay dung model cho RUN_ID duoc chon (B.1-B.2)
# Chi xay dung model cua run duoc chon, khong xay dung ca VGG16 va ResNet-18.
# ============================================================
utils.set_seed(SEED)

spec = RUN_SPECS[RUN_ID]

# Trong reuse, khoi tao kien truc khong tai weights ImageNet; checkpoint se duoc nap sau.
LOAD_PRETRAINED = (RUN_MODE == "train")

cfg_selected = ModelConfig(
    arch=spec["arch"],
    num_classes=NUM_CLASSES,
    weights=spec["weights"] if LOAD_PRETRAINED else None,
    freeze_backbone=spec["freeze_backbone"],
    unfreeze_last_n=spec["unfreeze_last_n"],
)
model_selected = build_model(cfg_selected)

# Hien thi kien truc bang torchinfo.summary
print("=" * 60)
print(f"{RUN_ID} | {spec['arch']} | weights={spec['weights'] if LOAD_PRETRAINED else 'None'} | freeze_backbone={spec['freeze_backbone']}")
print("=" * 60)
torchinfo_summary(
    model=model_selected,
    input_size=(BATCH_SIZE, 3, 224, 224),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"],
)

# Verify output shape: phai la [B, NUM_CLASSES] cho CIFAR-10
dummy = torch.randn(2, 3, 224, 224)
with torch.inference_mode():
    out_selected = model_selected(dummy)
assert out_selected.shape == (2, NUM_CLASSES), f"{RUN_ID} shape sai: {out_selected.shape}"
print(f"{RUN_ID} output: {out_selected.shape}")

# Hien thi lop classifier da thay
print(f'{RUN_ID} classifier (head):')
print(model_selected.head)


---
## 3. CIFAR-10 Model Adaptation

Phần này minh họa thay lớp classifier để tạo đầu ra 10 lớp, đồng봉 backbone và giữ BatchNorm ở trạng thái phù hợp khi train.

| Chiến lược | Nghĩa |
|---|---|
| **Feature extraction** | Đóng băng backbone, chỉ cập nhật classifier mới |
| **Fine-tuning** | Mở khóa một hoặc nhiều block cuối và cập nhật cùng classifier |
| **Full training** | Cập nhật toàn bộ mạng; không phải cấu hình của các run Phase 8 |

Các cell model hiện nay đã điều khiển bởi `RUN_SPECS[RUN_ID]`; chỉ xây dựng model của run được chọn.



---
## 4. Training & Validation — chỉ train/reuse cho RUN_ID được chọn

Cell này train hoặc reuse cho `RUN_ID` được chọn ở cell cấu hình. Không tự chạy thêm run khác.

**Không chạy training lại nếu artifact đã đầy đủ (nếu `RESUME=False`).** Việc tách hợp train/resume/reuse theo từng `RUN_ID` sẽ thuộc phase thực nghiệm sau.



In [ ]:
def log_run_config(writer, config: dict):
    """Ghi run config vào TensorBoard text tab."""
    lines = [f'`{k}` = `{v}`' for k, v in config.items()]
    writer.add_text('run_config', '  \n'.join(lines), global_step=0)

loss_fn = nn.CrossEntropyLoss()

---
### 4.1. Train/Reuse cho RUN_ID được chọn

Huấn luyện hoặc nạp lại artifact cho `RUN_ID` qua `engine.train`.



In [ ]:
# ============================================================
# Train hoac reuse cho RUN_ID duoc chon (B.3-B.4)
# Chi chay nhánh tuong ung voi RUN_ID, khong tu chay them run khac.
# engine.py phu troch vong lap train/validation, chon best model va checkpoint/resume.
# ============================================================
if RUN_MODE == "train":
    utils.set_seed(SEED)

    # Xay dung optimizer theo spec
    if spec["param_groups"] is not None:
        # Fine-tune: param groups voi LR rieng cho head va backbone
        from going_modular.model_builder import get_param_groups
        param_groups = get_param_groups(model_selected, cfg_selected,
                                        spec["param_groups"]["head"],
                                        spec["param_groups"]["backbone"])
        optimizer_selected = torch.optim.Adam(param_groups, lr=LR)
    else:
        optimizer_selected = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model_selected.parameters()), lr=LR
        )

    loss_fn = nn.CrossEntropyLoss()

    # Train-mode callback: giu BatchNorm cua backbone o eval khi freeze
    train_mode_fn = None
    if spec["arch"] == "resnet18" and spec["freeze_backbone"]:
        train_mode_fn = lambda m: set_train_mode(m, cfg_selected)

    # TensorBoard writer
    writer_selected = SummaryWriter(log_dir=str(utils.make_session_log_dir(LOG_ROOT, RUN_ID, SESSION_ID)))

    RUN_CONFIG = {
        "run_id": RUN_ID,
        "model": spec["arch"],
        "weights": spec["weights"] if LOAD_PRETRAINED else None,
        "data": {
            "transform_model": spec["arch"],
            "transform_weights": TRANSFORM_WEIGHTS,
        },
        "num_classes": NUM_CLASSES,
        "freeze_backbone": spec["freeze_backbone"],
        "unfreeze_last_n": spec["unfreeze_last_n"],
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "epochs": EPOCHS,
        "seed": SEED,
        "run_mode": RUN_MODE,
        "resume": RESUME,
    }

    start_epoch = 0
    print('-' * 55)
    print(f'{RUN_ID} | mode={RUN_MODE} | resume={RESUME} | start_epoch={start_epoch}')
    print('-' * 55)

    start_time = timer()
    try:
        history_selected = engine.train(
            model=model_selected,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_selected,
            loss_fn=loss_fn,
            epochs=EPOCHS,
            device=DEVICE,
            writer=writer_selected,
            save_path=str(BEST_MODEL_PATH),
            checkpoint_dir=str(RUN_ARTIFACT_DIR),
            resume=RESUME,
            expected_phase_config=RUN_CONFIG,
            start_epoch=start_epoch,
            train_mode_fn=train_mode_fn,
            verbose=False,
        )
    finally:
        writer_selected.close()

    train_time = timer() - start_time
    RUN_CONFIG['train_seconds'] = train_time
    utils.save_config(str(RUN_ARTIFACT_DIR), RUN_CONFIG)
    utils.save_history(str(RUN_ARTIFACT_DIR), history_selected)

    best_val_acc = max(history_selected['val_acc'])
    print(f'{RUN_ID} ready | train time: {train_time:.1f}s | best val_acc: {best_val_acc:.4f}')

else:
    # Reuse: nap model/history/config tu artifact, khong train lai
    history_selected, saved_config_selected = utils.load_experiment(str(RUN_ARTIFACT_DIR))
    utils.load_model(model_selected, str(BEST_MODEL_PATH), device=DEVICE)
    train_time = float(saved_config_selected.get('train_seconds', 0.0))
    best_val_acc = max(history_selected['val_acc'])
    print(f'{RUN_ID} reused | train time: {train_time:.1f}s | best val_acc: {best_val_acc:.4f}')
    print(f"Loaded saved config: {saved_config_selected}")

# Luu bien toan cuc de cac cell sau su dung
globals()["model_selected"] = model_selected
globals()["history_selected"] = history_selected if 'history_selected' in dir() else None
globals()["best_val_acc_selected"] = best_val_acc
globals()["train_time_selected"] = train_time
globals()["RUN_CONFIG_selected"] = RUN_CONFIG if 'RUN_CONFIG' in dir() else saved_config_selected



---
### 4.2. Cấu hình BatchNorm khi freeze backbone

Khi `freeze_backbone=True`, `set_train_mode` giữ BatchNorm của backbone ở `eval` trong toàn bộ quá trình train.



---
## 5. Model Comparison — sau khi có kết quả

Phần này dùng để kiểm so sánh history, loss/accuracy curves và thời gian giữa các run sau khi hoàn thành. Các biểu đồ vẽ chỉ cho `RUN_ID` đã chọn.

Không chạy phần so sánh như một bước kiểm chứng cấu hình Phase 8.



In [ ]:
# ============================================================
# Hien thi ket qua chi RUN_ID duoc chon (C)
# Neu chua co ket qua (tu cell train/reuse), bao bo thay hien thi gia tri placeholder.
# ============================================================
if 'history_selected' in globals() and history_selected is not None:
    summary_data = {
        'Run': [RUN_ID],
        'Best val_acc': [f'{best_val_acc_selected:.4f}'],
        'Train time (s)': [f'{train_time_selected:.1f}'],
        'Trainable params': [f"{count_params(model_selected)['trainable']:,}"],
    }
    print(pd.DataFrame(summary_data).set_index('Run'))
else:
    print(f"Chua co ket qua training/reuse cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi xem tóm tắt.")



In [ ]:
# ============================================================
# Ve loss/accuracy curves cho RUN_ID duoc chon (C)
# ============================================================
if 'history_selected' in globals() and history_selected is not None:
    evaluate.plot_history(
        histories={RUN_ID: history_selected},
        save_path=str(EVAL_OUTPUT_DIR / "history.png"),
        show=True,
    )
else:
    print(f"Chua co lich su training/reuse cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi ve bieu do.")



---
## 6. ResNet-18 Fine-tuning — phụ thuộc baseline

Run fine-tune cần khởi tạo từ `best_model.pth` của `v1_resnet18`, sau đó mở khóa `layer4` và classifier với optimizer/LR theo cấu hình đã chốt.

Cell hiện tại là pipeline fine-tune cố định, chưa được điều khiển bởi `RUN_ID`, `RUN_SPECS` và `RESUME`. Chỉ thực hiện ở phase huấn luyện sau khi baseline hoàn tất và cell cần được tách hợp.



---
## 7. Final Test Evaluation — chỉ chạy sau khi chốt mô hình

> **Chỉ chạy sau khi quá trình huấn luyện và lựa chọn checkpoint bằng validation đã hoàn tất.** Không sử dụng kết quả test để chọn kiến trúc, hyperparameter hay checkpoint.

Phần này tạo metrics, confusion matrix, per-class accuracy và vẽ dự đoán sai. Đây là đánh giá cuối của thực nghiệm, không thuộc Phase 8 cấu hình notebook.



In [ ]:
# ============================================================
# Danh gia test cho RUN_ID duoc chon (C)
# Chi danh gia khi run da hoan thanh; luu/nap test_metrics.json theo RUN_ID.
# ============================================================
if 'model_selected' in globals():
    inf_selected = run_inference(model_selected, test_dl, DEVICE, loss_fn=loss_fn)
    result_selected = evaluate_predictions(inf_selected, CLASS_NAMES)

    # Luu test_metrics.json neu chua co
    if not (EVAL_OUTPUT_DIR / "test_metrics.json").is_file():
        utils.save_test_metrics(str(EVAL_OUTPUT_DIR), result_selected)
        print(f"Da luu test_metrics.json: {EVAL_OUTPUT_DIR / "test_metrics.json"}")
    else:
        # Neu da co, nap lai de tranh tinh lai
        cached = utils.load_test_metrics(str(EVAL_OUTPUT_DIR))
        print(f"test_metrics.json da ton tai, su dung ban luu: {cached}")

    print('=== TEST SET RESULTS ===')
    print(f"Run: {RUN_ID}")
    for k, v in result_selected.items():
        print(f"  {k}: {v}")
else:
    print(f"Chua xay dung model cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi danh gia test.")



In [ ]:
# ============================================================
# Confusion matrix cho RUN_ID duoc chon (C)
# ============================================================
if 'model_selected' in globals():
    y_pred_sel, y_true_sel = evaluate.predict_dataloader(model_selected, test_dl, device=DEVICE)
    cm_sel = evaluate.compute_confusion_matrix(y_true=y_true_sel, y_pred=y_pred_sel, num_classes=NUM_CLASSES)

    evaluate.plot_confusion_matrix(
        cm_sel,
        class_names=CLASS_NAMES,
        title=f'Confusion Matrix — {RUN_ID} (Test Set)',
        save_path=str(EVAL_OUTPUT_DIR / "confusion_matrix.png"),
        show=True,
    )
else:
    print(f"Chua xay dung model cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi ve confusion matrix.")



In [ ]:
# ============================================================
# Per-class accuracy cho RUN_ID duoc chon (C)
# ============================================================
if 'model_selected' in globals():
    y_pred_sel, y_true_sel = evaluate.predict_dataloader(model_selected, test_dl, device=DEVICE)
    cm_sel = evaluate.compute_confusion_matrix(y_true=y_true_sel, y_pred=y_pred_sel, num_classes=NUM_CLASSES)
    class_acc_sel = evaluate.per_class_accuracy(cm_sel)

    df_class_sel = pd.DataFrame({
        'Class': CLASS_NAMES,
        'Accuracy': [f'{a:.3f}' for a in class_acc_sel.tolist()],
    }).sort_values('Accuracy')

    print('Per-class accuracy (thap -> cao):')
    print(df_class_sel.to_string(index=False))
else:
    print(f"Chua xay dung model cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi tinh per-class accuracy.")



In [ ]:
# ============================================================
# Top 5 nham lan cho RUN_ID duoc chon (C)
# ============================================================
if 'model_selected' in globals():
    y_pred_sel, y_true_sel = evaluate.predict_dataloader(model_selected, test_dl, device=DEVICE)
    cm_sel = evaluate.compute_confusion_matrix(y_true=y_true_sel, y_pred=y_pred_sel, num_classes=NUM_CLASSES)
    top5_sel = evaluate.top_confusions(cm_sel, CLASS_NAMES, k=5)
    print('Top 5 nham lan:')
    print(f'  {"True":>12} -> {"Predicted":<12}  Count')
    print('  ' + '-' * 38)
    for true_cls, pred_cls, count in top5_sel:
        print(f'  {true_cls:>12} -> {pred_cls:<12}  {count}')
else:
    print(f"Chua xay dung model cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi tinh top confusions.")



In [ ]:
# ============================================================
# Visualize du doan sai cho RUN_ID duoc chon (C)
# ============================================================
if 'model_selected' in globals():
    y_pred_sel, y_true_sel = evaluate.predict_dataloader(model_selected, test_dl, device=DEVICE)
    wrong_mask_sel = (y_pred_sel != y_true_sel)
    wrong_indices_sel = wrong_mask_sel.nonzero(as_tuple=True)[0][:9]

    all_imgs, all_true = [], []
    for imgs, lbls in test_dl:
        all_imgs.append(imgs)
        all_true.append(lbls)
    all_imgs = torch.cat(all_imgs)

    sample_imgs = [denormalize(all_imgs[i]) for i in wrong_indices_sel]
    sample_true = [y_true_sel[i].item() for i in wrong_indices_sel]
    sample_pred = [y_pred_sel[i].item() for i in wrong_indices_sel]

    evaluate.plot_sample_predictions(
        images=sample_imgs,
        true_labels=sample_true,
        pred_labels=sample_pred,
        class_names=CLASS_NAMES,
        ncols=3,
        save_path=str(EVAL_OUTPUT_DIR / "sample_predictions.png"),
        show=True,
    )
else:
    print(f"Chua xay dung model cho RUN_ID='{RUN_ID}'. Chay cell train/tren truoc khi visualize du doan sai.")



---
## Trạng thái và bước tiếp theo

### Phase 8 — cấu hình notebook

- Chuẩn hóa nguồn code, đường dẫn Drive/runtime, dataset và cấu hình `RUN_ID`.
- Kiểm tra thiết lập cấu hình mà không huấn luyện hay đánh giá test.
- Chưa xác nhận bootstrap/đọc-ghi Drive thật cho đến khi người dùng thực hiện trên Colab.

### Phase thực nghiệm tiếp theo

Sau khi các cell train/resume/reuse được nối với cấu hình sáu run và các đường dẫn artifacts, người dùng chạy từng run theo thứ tự/phụ thuộc trong kế hoạch. Đánh giá test chỉ thực hiện sau khi mỗi run hoàn tất.

